In [1]:
from langchain_ollama import OllamaLLM
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.retrievers import EnsembleRetriever
from langchain.chains import RetrievalQA

import os

- RAG retrievers (via ChromaDB library)

In [39]:
import chromadb

# Initialize the persistent ChromaDB client
chroma_client = chromadb.PersistentClient(path="../output/vectorstore")


# List all collections in the database
collections = chroma_client.list_collections()
print("Collections in the database:")
for collection in collections:
    # Print the collection name directly since collection is now a string
    print(collection)

# Delete a specific collection if needed
# chroma_client.delete_collection(name="base_address")

# If you need to work with a specific collection, use get_collection():
address = chroma_client.get_collection(name="base_address")

# show the number of documents in the collection
print(address.count())



Collections in the database:
base_address
27800


- Similarity Search (via ChromaDB library)

In [238]:
result = \
address.query(
    query_texts=[("BANDAR WARISAN PUTERI JENIS AMETHYST,AMPANGAN,SEREMBAN,NEGERI SEMBILAN").upper()], # Chroma will embed this for you
    n_results=1 # how many results to return
)
result

{'ids': [['e55c5a1c-676d-461d-a8a9-61e17bb28641']],
 'embeddings': None,
 'documents': [['BANDAR SEREMBAN, 70000, SEREMBAN, NEGERI SEMBILAN']],
 'uris': None,
 'data': None,
 'metadatas': [[{'district': 'SEREMBAN',
    'row': 1526,
    'source': '../output/address.csv',
    'state': 'NEGERI SEMBILAN'}]],
 'distances': [[0.5250101685523987]],
 'included': [<IncludeEnum.distances: 'distances'>,
  <IncludeEnum.documents: 'documents'>,
  <IncludeEnum.metadatas: 'metadatas'>]}

In [44]:
print(f"Consine Similarity Score: {result['distances'][0][0]}") # lower is better
print(f"{result['documents'][0][0]}")

Consine Similarity Score: 0.5250101685523987
BANDAR SEREMBAN, 70000, SEREMBAN, NEGERI SEMBILAN


- RAG retrivers (via Langchain library)

In [45]:

# Initialize the embedding
#oembed = OllamaEmbeddings(base_url="http://localhost:11434", model="llama3.2:latest") # 3072-dim
hfembed = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")  # 384-dim

# Connect to existing vectorstore
vectorstore = Chroma(
    collection_name="base_address",
    embedding_function=hfembed,
    persist_directory="../output/vectorstore"
)

C:\Users\izardy\AppData\Local\Temp\ipykernel_34676\2820794496.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  hfembed = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")  # 384-dim
c:\Users\izardy\AppData\Local\miniconda3\envs\etl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [47]:
query = "BANDAR WARISAN PUTERI JENIS AMETHYST,AMPANGAN,SEREMBAN,NEGERI SEMBILAN"

# Generate embeddings for the query (depends on your embedding setup)
query_embedding = hfembed.embed_query(query.upper())

# Perform the query using the generated embedding
results = address.query(
    query_embeddings=[query_embedding],
    n_results=5  # adjust based on how many matches you want
)

# Show the result , lower cosine similarity score is better
for distance, doc in zip(results['distances'][0], results['documents'][0]):
    print(f"{distance:.4f} → {doc}")



0.5250 → BANDAR SEREMBAN, 70000, SEREMBAN, NEGERI SEMBILAN
0.5388 → BANDAR SEREMBAN UTAMA, 70000, SEREMBAN, NEGERI SEMBILAN
0.5877 → BANDAR NILAI UTAMA, 71800, SEREMBAN, NEGERI SEMBILAN
0.6244 → AMPANGAN, 70400, SEREMBAN, NEGERI SEMBILAN
0.6424 → BANDAR SEREMBAN 3, 70300, SEREMBAN, NEGERI SEMBILAN


- Calculation using SkLearn

In [51]:
query1 = "BANDAR WARISAN PUTERI JENIS AMETHYST,AMPANGAN,SEREMBAN,NEGERI SEMBILAN"
# Initialize the embedding
embed1=hfembed.embed_query((query1.upper()))

query2 = "AMPANGAN, 70400, SEREMBAN, NEGERI SEMBILAN"
embed2=hfembed.embed_query(query2)

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

sim_score = cosine_similarity(
    np.array(embed1).reshape(1, -1),
    np.array(embed2).reshape(1, -1)
)

print("Cosine similarity:", sim_score[0][0])


Cosine similarity: 0.6878122067899873


c:\Users\izardy\AppData\Local\miniconda3\envs\etl\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [58]:
vectorstore.search("BANDAR WARISAN PUTERI JENIS AMETHYST,AMPANGAN,SEREMBAN,NEGERI SEMBILAN",k=5,search_type="similarity")

[Document(id='e55c5a1c-676d-461d-a8a9-61e17bb28641', metadata={'district': 'SEREMBAN', 'row': 1526, 'source': '../output/address.csv', 'state': 'NEGERI SEMBILAN'}, page_content='BANDAR SEREMBAN, 70000, SEREMBAN, NEGERI SEMBILAN'),
 Document(id='0a8a14a7-60e5-4fad-86bf-5687f05b59e5', metadata={'district': 'SEREMBAN', 'row': 1528, 'source': '../output/address.csv', 'state': 'NEGERI SEMBILAN'}, page_content='BANDAR SEREMBAN UTAMA, 70000, SEREMBAN, NEGERI SEMBILAN'),
 Document(id='2214a3a8-4f55-40f1-9e13-4b4ef5481898', metadata={'district': 'SEREMBAN', 'row': 1532, 'source': '../output/address.csv', 'state': 'NEGERI SEMBILAN'}, page_content='BANDAR NILAI UTAMA, 71800, SEREMBAN, NEGERI SEMBILAN'),
 Document(id='0cf2a737-1b9f-4971-a8f6-8feb1e90231e', metadata={'district': 'SEREMBAN', 'row': 1518, 'source': '../output/address.csv', 'state': 'NEGERI SEMBILAN'}, page_content='AMPANGAN, 70400, SEREMBAN, NEGERI SEMBILAN'),
 Document(id='59dde7f1-a95f-40ce-a990-2700ff44b003', metadata={'district':

In [106]:
results = vectorstore.similarity_search_with_score(str.upper("BANDAR WARISAN PUTERI JENIS AMETHYST,AMPANGAN,SEREMBAN,NEGERI SEMBILAN"), k=5)
for doc, score in results:
    print(f"{score:.4f} → {doc.page_content}")


0.5250 → BANDAR SEREMBAN, 70000, SEREMBAN, NEGERI SEMBILAN
0.5388 → BANDAR SEREMBAN UTAMA, 70000, SEREMBAN, NEGERI SEMBILAN
0.5877 → BANDAR NILAI UTAMA, 71800, SEREMBAN, NEGERI SEMBILAN
0.6244 → AMPANGAN, 70400, SEREMBAN, NEGERI SEMBILAN
0.6424 → BANDAR SEREMBAN 3, 70300, SEREMBAN, NEGERI SEMBILAN


- Combine RAG Result & LLM

In [61]:
# Initialize the LLM
llm = OllamaLLM(model="llama3.2:latest", base_url="http://localhost:11434")

In [83]:
# Test LLM
llm.invoke("Hello world")

"Hello! It's nice to meet you. Is there something I can help you with or would you like to chat?"

In [242]:
import re
import json


postcode_matcher="GRN237447 LOT11499 SEKSYEN 1 (HSD16524,PT1560), TAMAN KASIH PUTERA,PEKAN BAHAU, JEMPOL, NEGERI SEMBILAN,PEKAN BAHAU,JEMPOL"

# Generate embeddings for the query (depends on your embedding setup)
query_embedding = hfembed.embed_query(postcode_matcher.upper())

# Perform the query using the generated embedding
results = address.query(
    query_embeddings=[query_embedding],
    n_results=5  # adjust based on how many matches you want
)

max_attempts = 3
attempt = 0
postcode = None

while attempt < max_attempts and postcode is None:
    answer = llm.invoke(
        f"Given the following informations {results['documents'][0][0]} ; {results['documents'][0][1]} ; {results['documents'][0][2]} ; \
        {results['documents'][0][3]} ; {results['documents'][0][4]} for evaluation. \
        What is the possible postcode for {postcode_matcher} strictly based on the informations?\
        Answer 1 postcode only in json format"
    )
    json_match = re.search(r"\{[\s\S]*?\}", answer)
    if json_match:
        extracted_json = json_match.group(0)
        postcode = json.loads(extracted_json)["postcode"]
        print(postcode)
    else:
        print("No JSON found in answer. Retrying...")
        attempt += 1

if postcode is None:
    print("Failed to extract JSON after multiple attempts.")


c:\Users\izardy\AppData\Local\miniconda3\envs\etl\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


72100


In [1]:
import sys
sys.path.append("..")  # or the actual path to the script
from postcode import malaysia_postcode

print(malaysia_postcode("GRN237447 LOT11499 SEKSYEN 1 (HSD16524,PT1560), TAMAN KASIH PUTERA,PEKAN BAHAU, JEMPOL, NEGERI SEMBILAN,PEKAN BAHAU,JEMPOL"))

c:\Users\izardy\Documents\GitHub\postcode-district-subdistrict-mapper\notebook\..\postcode.py:16: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  hfembed = HuggingFaceEmbeddings(model_name=embedding_model)
c:\Users\izardy\AppData\Local\miniconda3\envs\etl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\izardy\AppData\Local\miniconda3\envs\etl\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in versio

72100
